# Spornado Disease Risk — Prediction Model Demo

**Purpose**: Demonstrate the production training pipeline and inference API using `src/`.  
**Audience**: Engineers and data scientists reviewing or extending the model.  
**Relation to production code**: This notebook *calls the same code that runs in production*. It does **not** reimplement the pipeline — all logic lives in `src/`.

## Contents
1. Setup & Configuration
2. Feature Engineering Preview
3. Train Per-Crop Models
4. Model Performance Summary
5. Threshold Visualisation
6. Single-Observation Prediction
7. Batch Prediction

### Pipeline recap
```
Raw CSV
  src.features.build_features()       <- normalise, dates, rolling stats
    src.build_crop_models.train_all_crops()
      TimeSeriesSplit CV + Pipeline(Imputer->Scaler->LogisticRegression)
        find_optimal_threshold()       <- recall-constrained, not 0.5
          models/*.pkl + models/*_thresholds.json
            src.predict.Predictor.predict()  <- inference
```

> **Recall constraint**: The deployment threshold is chosen to catch >= 90% of real outbreaks
> (configurable via `model.min_recall` in `config/config.yaml`).
> A missed outbreak costs a crop; a false alarm costs a spray — asymmetric costs
> drive a lower-than-0.5 threshold.

## 1. Setup & Configuration

In [ ]:
import sys, warnings, json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# Project root on sys.path (works from notebooks/ or project root)
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config_loader import load as load_config
from src.features import build_features, get_feature_columns, TARGET_COL
from src.build_crop_models import train_all_crops, risk_label, find_optimal_threshold
from src.predict import Predictor

cfg = load_config(PROJECT_ROOT / 'config' / 'config.yaml')
DATA_RAW   = PROJECT_ROOT / cfg['paths']['data_merged']
MODELS_DIR = PROJECT_ROOT / cfg['paths']['models_dir']
DIR_OUT    = PROJECT_ROOT / 'outputs'
DIR_OUT.mkdir(exist_ok=True)

print('Config loaded.')
print(f'  min_recall         : {cfg["model"]["min_recall"]}')
print(f'  medium_band_factor : {cfg["model"]["medium_band_factor"]}')
print(f'  cv_folds           : {cfg["model"]["cv_folds"]}')
print(f'  Data path          : {DATA_RAW}')
print(f'  Models dir         : {MODELS_DIR}')


## 2. Feature Engineering Preview

All transformations are handled by `src.features.build_features()`.
This cell loads the raw CSV and runs the full feature pipeline so we can
inspect the resulting feature matrix before training.

> **Spore count is excluded from features.**  Using it would be target leakage — the device labels a trap positive/negative
> *based on* spore detection.  Weather signals alone allow predictions
> *before* a trap is read.

In [ ]:
raw = pd.read_csv(DATA_RAW, low_memory=False)
print(f'Raw shape: {raw.shape}')

df = build_features(raw, cfg)
print(f'After feature engineering: {df.shape}')
print()
print('Target balance (all crops):')
print(df[TARGET_COL].value_counts().rename({0: 'negative', 1: 'positive'}).to_string())


In [ ]:
feature_cols = get_feature_columns(cfg, df)
weather  = cfg['features']['weather']
temporal = cfg['features']['temporal']
rolling  = [c for c in feature_cols if 'roll' in c]

print(f'Total features : {len(feature_cols)}')
print(f'  Weather  ({len(weather):2d}) : {weather}')
print(f'  Temporal ({len(temporal):2d}) : {temporal}')
print(f'  Rolling  ({len(rolling):2d}) : {rolling[:4]} ...')
print()
print(df[feature_cols].describe().round(2))


## 3. Train Per-Crop Models

Calls `src.build_crop_models.train_all_crops()` which:
- Splits data chronologically (no future leakage)
- Runs 5-fold `TimeSeriesSplit` cross-validation on the training set
- Fits `Pipeline(SimpleImputer -> StandardScaler -> LogisticRegression)`
- Finds the recall-constrained threshold via `precision_recall_curve`
- Saves `.pkl` and `_thresholds.json` for each crop

> **Runtime**: ~10-30 seconds per crop on a laptop (n_jobs=1).

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)  # suppress INFO during notebook run

results_df = train_all_crops(
    data_path=str(DATA_RAW),
    cfg=cfg,
)

display_cols = [
    'crop', 'train_samples', 'test_samples',
    'cv_auc_mean', 'cv_auc_std',
    'test_auc', 'test_f1', 'test_precision', 'test_recall',
    'threshold_optimal', 'threshold_achieved_recall',
]
print(results_df[display_cols].to_string(index=False))


## 4. Model Performance Summary

Key distinction: **all metrics are computed at the deployment threshold**
(not the sklearn default of 0.5).  This is what matters in production.

| Metric | Meaning |
|---|---|
| `test_auc` | Threshold-independent discrimination (ROC area) |
| `test_recall` | Fraction of real outbreaks caught — must be >= `min_recall` |
| `test_precision` | Fraction of HIGH alerts that are real outbreaks |
| `threshold_optimal` | The actual cut-off used at inference time |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(
    'Per-Crop Model Performance (metrics at deployment threshold)',
    fontsize=13, fontweight='bold'
)

metrics = ['test_auc', 'test_recall', 'test_precision']
labels  = ['ROC AUC', 'Recall (outbreak catch rate)', 'Precision (alert accuracy)']
colors  = ['#2980b9', '#e74c3c', '#27ae60']

for ax, metric, label, color in zip(axes, metrics, labels, colors):
    bars = ax.bar(results_df['crop'], results_df[metric],
                  color=color, alpha=0.8, edgecolor='black')
    ax.set_title(label)
    ax.set_ylim(0, 1.08)
    ax.set_ylabel('Score')
    if metric == 'test_recall':
        ax.axhline(cfg['model']['min_recall'], color='red', linestyle='--',
                   linewidth=1.5, label=f'min_recall={cfg["model"]["min_recall"]}')
        ax.legend(fontsize=8)
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(DIR_OUT / '07_model_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 07_model_performance.png')


## 5. Threshold Visualisation

Each crop has three risk zones derived from the recall-constrained threshold:

```
  0.0 ---- low_max --------- high_min (= optimal) ---- 1.0
     LOW (safe)  MEDIUM (uncertain)  HIGH (alert)
```

`low_max = optimal x medium_band_factor`
The uncertain zone is proportional to the decision boundary, not a fixed offset.

In [ ]:
ncrop = len(results_df)
fig, axes = plt.subplots(1, ncrop, figsize=(5 * ncrop, 4))
if ncrop == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, results_df.iterrows()):
    low_max  = row['threshold_low_max']
    high_min = row['threshold_high_min']  # == threshold_optimal

    ax.barh([''], [low_max],             left=0,        color='#2ecc71', alpha=0.7, label='LOW')
    ax.barh([''], [high_min - low_max],  left=low_max,  color='#f39c12', alpha=0.7, label='MEDIUM')
    ax.barh([''], [1 - high_min],        left=high_min, color='#e74c3c', alpha=0.7, label='HIGH')
    ax.axvline(high_min, color='#c0392b', linewidth=2.5)
    ax.axvline(low_max,  color='#27ae60', linewidth=1.5, linestyle='--')
    ax.set_xlim(0, 1)
    ax.set_xlabel('Risk probability')
    ax.set_title(
        f"{row['crop']}\n"
        f"Threshold={high_min:.3f}  Recall={row['threshold_achieved_recall']:.1%}"
        f"  Precision={row['test_precision']:.1%}",
        fontsize=10
    )
    ax.legend(fontsize=8, loc='upper right')

fig.suptitle('Risk Zones per Crop (recall-constrained thresholds)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 6. Single-Observation Prediction

`src.predict.Predictor` loads the saved model and threshold JSON at construction time,
then returns a structured dict with the risk probability, label, and diagnostic metadata.
Missing features are filled by the `SimpleImputer` inside the Pipeline using training-set means.

In [ ]:
predictor = Predictor(cfg)

result = predictor.predict(
    crop='Corn',
    temperature_max_c=28.5,
    temperature_min_c=14.0,
    temperature_mean_c=21.0,
    humidity_max_percent=83.0,
    precipitation_mm=2.1,
    wind_speed_max_kmh=18.0,
    dew_point_min_c=12.5,
    month=8,
    day_of_year=220,
    week_of_year=32,
)

print(json.dumps(result, indent=2))


## 7. Batch Prediction

`Predictor.predict_batch()` accepts a DataFrame with a `crop_type` column
and all required weather / temporal feature columns, and appends
`risk_probability` and `risk_label` columns.

In [ ]:
batch = pd.DataFrame([
    {'crop_type': 'Corn',    'temperature_max_c': 28.5, 'humidity_max_percent': 83.0,
     'precipitation_mm': 2.1, 'month': 8,  'day_of_year': 220, 'week_of_year': 32},
    {'crop_type': 'Corn',    'temperature_max_c': 18.0, 'humidity_max_percent': 55.0,
     'precipitation_mm': 0.0, 'month': 6,  'day_of_year': 160, 'week_of_year': 23},
    {'crop_type': 'Soybean', 'temperature_max_c': 31.0, 'humidity_max_percent': 90.0,
     'precipitation_mm': 5.5, 'month': 7,  'day_of_year': 200, 'week_of_year': 29},
    {'crop_type': 'Potato',  'temperature_max_c': 22.0, 'humidity_max_percent': 78.0,
     'precipitation_mm': 1.2, 'month': 9,  'day_of_year': 252, 'week_of_year': 36},
])

scored = predictor.predict_batch(batch, crop_col='crop_type')
print(scored[['crop_type', 'temperature_max_c', 'humidity_max_percent',
              'risk_probability', 'risk_label']].to_string(index=False))
